In [1]:
from copy import deepcopy
import torch
import sys
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda import amp
from spikingjelly.activation_based import functional, surrogate, neuron, layer
from spikingjelly.activation_based.model import parametric_lif_net
from spikingjelly.datasets.dvs128_gesture import DVS128Gesture
from torch.utils.data import DataLoader
import time
import os
import argparse
import datetime

In [2]:
torch.manual_seed(1)

In [3]:
T = 16
b = 8
j = 8
lr = 0.01
epochs = 20
channels = 16

data_dir = os.path.expanduser('~/datasets/DVSGesture/')

In [4]:
device = 'cuda:0'

In [5]:
class DVSGestureNet(nn.Module):
    def __init__(self, channels=32, spiking_neuron: callable=None, is_seperable=False, kernel_size=3, **kwargs):
        super().__init__()

        conv = []
        ## Stem
        conv.append(layer.Conv2d(2, channels, kernel_size=2, stride=2, 
                                    bias=False))
        conv.append(layer.BatchNorm2d(channels))

        ## Middle Layers
        for i in range(4):
            if is_seperable:
                conv.append(layer.Conv2d(channels, channels,
                                         kernel_size=kernel_size, groups=channels,
                                         padding='same', bias=False)
                )
                conv.append(layer.Conv2d(channels, channels, kernel_size=1))
            else:
                conv.append(layer.Conv2d(channels, channels, kernel_size=kernel_size, 
                                         padding='same', bias=False))
            conv.append(layer.BatchNorm2d(channels))    
            conv.append(spiking_neuron(**deepcopy(kwargs)))
            if i != 3:
                conv.append(layer.Conv2d(channels, 2*channels, kernel_size=2, bias=False, stride=2))
                channels = channels * 2
        

        self.conv = nn.Sequential(
            *conv, 
            layer.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = layer.Linear(in_features=channels, out_features=11)
        
    def forward(self, x: torch.Tensor):
        x = self.conv(x).mean((3, 4))
        x = self.fc(x)
            
        return x


In [6]:
train_set = DVS128Gesture(root=data_dir, train=True, data_type='frame', frames_number=T, split_by='number')
test_set = DVS128Gesture(root=data_dir, train=False, data_type='frame', frames_number=T, split_by='number')

The directory [/home/tahaf/datasets/DVSGesture/frames_number_16_split_by_number] already exists.
The directory [/home/tahaf/datasets/DVSGesture/frames_number_16_split_by_number] already exists.


In [7]:
train_data_loader = torch.utils.data.DataLoader(
    dataset=train_set,
    batch_size=b,
    shuffle=True,
    drop_last=True,
    num_workers=j,
    pin_memory=True
)

test_data_loader = torch.utils.data.DataLoader(
    dataset=test_set,
    batch_size=b,
    shuffle=True,
    drop_last=False,
    num_workers=j,
    pin_memory=True
)

In [8]:
scaler = amp.GradScaler()

In [9]:
def check_model(kernel_size, channels, is_seperable):
    print(f'Results for kernel size = {kernel_size} and seperable convolution = {is_seperable} and channels = {channels}')
    max_test_acc = -1

    net = DVSGestureNet(
        channels=channels,
        kernel_size=kernel_size,
        spiking_neuron=neuron.LIFNode,
        surrogate_function=surrogate.ATan(),
        is_seperable=is_seperable,
        detach_reset=True
    )
    net.to(device)
    num_params = sum(p.numel() for p in net.parameters())
    functional.set_step_mode(net, step_mode='m')

    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, total_steps=round(epochs*60000/b), max_lr=lr)
    
    start_time = time.time()
    for epoch in range(epochs):
        net.train()
        train_loss = 0
        train_acc = 0
        train_samples = 0
        for frame, label in train_data_loader:
            optimizer.zero_grad()
            frame = frame.to(device)
            frame = frame.transpose(0, 1)  # [N, T, C, H, W] -> [T, N, C, H, W]
            label = label.to(device)
            label_onehot = F.one_hot(label, 11).float()
    
            if scaler is not None:
                with amp.autocast():
                    out_fr = net(frame)
                    loss = functional.temporal_efficient_training_cross_entropy(out_fr, label)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out_fr = net(frame)
                loss = functional.temporal_efficient_training_cross_entropy(out_fr, label)
                loss.backward()
                optimizer.step()
    
            train_samples += label.numel()
            train_loss += loss.item() * label.numel()
            train_acc += (out_fr.mean(0).argmax(1) == label).float().sum().item()
    
            functional.reset_net(net)
    
        train_loss /= train_samples
        train_acc /= train_samples
    
        lr_scheduler.step()
    
        net.eval()
        test_loss = 0
        test_acc = 0
        test_samples = 0
        with torch.no_grad():
            for frame, label in test_data_loader:
                frame = frame.to(device)
                frame = frame.transpose(0, 1)  # [N, T, C, H, W] -> [T, N, C, H, W]
                label = label.to(device)
                label_onehot = F.one_hot(label, 11).float()
                out_fr = net(frame)
                loss = functional.temporal_efficient_training_cross_entropy(out_fr, label)
                test_samples += label.numel()
                test_loss += loss.item() * label.numel()
                test_acc += (out_fr.mean(0).argmax(1) == label).float().sum().item()
                functional.reset_net(net)
        test_loss /= test_samples
        test_acc /= test_samples
        max_test_acc = max(max_test_acc, test_acc)
        
        print(f'epoch = {epoch}, train_loss ={train_loss: .4f}, train_acc ={train_acc: .4f}, test_loss ={test_loss: .4f}, test_acc ={test_acc: .4f}, max_test_acc ={max_test_acc: .4f}')
    end_time = time.time()
    total_time = end_time - start_time
    print(f'total time: {total_time}s')
    print('-'*1000)

    return {
        'accuracy': max_test_acc,
        'total_time': total_time,
        'num_params': num_params,
    }

In [10]:
kernel_sizes = [3, 5, 7]
channels = [16, 32]
results = {}

In [11]:
for channel in channels:
    results[channel] = {}
    for kernel_size in kernel_sizes:
        results[channel][kernel_size] = {}
        for is_seperable in [False, True]:
            result = check_model(channels=channel, kernel_size=kernel_size, is_seperable=is_seperable)
            results[channel][kernel_size][is_seperable] = result

Results for kernel size = 3 and seperable convolution = False and channels = 16
epoch = 0, train_loss = 2.2789, train_acc = 0.2228, test_loss = 2.0839, test_acc = 0.2604, max_test_acc = 0.2604
epoch = 1, train_loss = 2.0039, train_acc = 0.4031, test_loss = 1.8610, test_acc = 0.4167, max_test_acc = 0.4167
epoch = 2, train_loss = 1.8140, train_acc = 0.4694, test_loss = 1.7418, test_acc = 0.4167, max_test_acc = 0.4167
epoch = 3, train_loss = 1.6493, train_acc = 0.5077, test_loss = 1.6192, test_acc = 0.4583, max_test_acc = 0.4583
epoch = 4, train_loss = 1.5068, train_acc = 0.5400, test_loss = 1.3348, test_acc = 0.5938, max_test_acc = 0.5938
epoch = 5, train_loss = 1.3907, train_acc = 0.5731, test_loss = 1.2820, test_acc = 0.5764, max_test_acc = 0.5938
epoch = 6, train_loss = 1.3060, train_acc = 0.5859, test_loss = 1.1791, test_acc = 0.6354, max_test_acc = 0.6354
epoch = 7, train_loss = 1.2520, train_acc = 0.6259, test_loss = 1.1709, test_acc = 0.6597, max_test_acc = 0.6597
epoch = 8, train

In [12]:
results

{16: {3: {False: {'accuracy': 0.8159722222222222,
    'total_time': 150.5706250667572,
    'num_params': 240907},
   True: {'accuracy': 0.7777777777777778,
    'total_time': 168.63015723228455,
    'num_params': 69227}},
  5: {False: {'accuracy': 0.8819444444444444,
    'total_time': 165.71714878082275,
    'num_params': 589067},
   True: {'accuracy': 0.84375,
    'total_time': 233.48447442054749,
    'num_params': 73067}},
  7: {False: {'accuracy': 0.8888888888888888,
    'total_time': 184.15782737731934,
    'num_params': 1111307},
   True: {'accuracy': 0.8090277777777778,
    'total_time': 353.7056257724762,
    'num_params': 78827}}},
 32: {3: {False: {'accuracy': 0.8402777777777778,
    'total_time': 255.19887280464172,
    'num_params': 959499},
   True: {'accuracy': 0.8368055555555556,
    'total_time': 266.39118790626526,
    'num_params': 267979}},
  5: {False: {'accuracy': 0.90625,
    'total_time': 277.0904760360718,
    'num_params': 2352139},
   True: {'accuracy': 0.923611

In [ ]:
import os
import json

with open(os.path.expanduser('~/new_results.json'), 'w') as f:
    json.dump(results, f, indent=6) 